# WSJ 2027 - Gruppindelning Direktresa

Assign confirmed direktresa deltagare into groups of exactly 36.

Direktresa participants travel independently to/from WSJ in Poland.
Same grouping algorithm as rundresa but applied to the direktresa subset.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — Hilbert curve sort
3. **Friend wish** — at least ONE friend in same group (soft goal)
4. **Max 8 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex balance

In [1]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

# Fetch and parse all participants
raw_data = u.fetch_participants()
df, skipped = u.build_participant_dataframe(raw_data)

Fetched 1972 participants
Confirmed: 1901, Unconfirmed: 33, Cancelled: 38
Total confirmed participants: 1900
Skipped: 33 unconfirmed, 1 wrong age/no DOB

By category:
category
deltagare    1585
ist           293
cmt            22

By travel type:
travel
rundresa      1257
direktresa     414
egen_resa      207
other           22

Friend wishes:
  With friend 1 member no: 1109
  With friend 2 member no: 656
  With friend 1 name (text): 86
  With friend 2 name (text): 60

Skipped participants:
  DELTAGARE wrong age: Ann-Ida Bergsten born 1983-07-25 (age 44)


In [2]:
GROUP_SIZE = 36

# Filter to direktresa deltagare only
df_direkt = df[df['travel'] == 'direktresa'].copy().reset_index(drop=True)
print(f"=== Direktresa deltagare ===")
print(f"Participants: {len(df_direkt)}")

# Assign coordinates and Hilbert sort
u.assign_coordinates(df_direkt)
df_sorted = u.add_hilbert_index(df_direkt)

n_full = len(df_sorted) // GROUP_SIZE
remainder = len(df_sorted) % GROUP_SIZE
total_groups = n_full + (1 if remainder > 0 else 0)
print(f"\nGroups: {n_full} x {GROUP_SIZE} + 1 x {remainder} = {total_groups} total")
print(f"\nBy region:")
print(df_sorted['region'].value_counts().to_string())
print(f"\nBy age:")
print(df_sorted['age'].value_counts().sort_index().to_string())
print(f"\nBy sex:")
print(df_sorted['sex'].map(u.SEX_MAP).value_counts().to_string())

=== Direktresa deltagare ===
Participants: 414
With coordinates: 354
Without coordinates (Sweden centroid): 60

Groups: 11 x 36 + 1 x 18 = 12 total

By region:
region
Region Stockholm    168
Region Södra         71
Region Norr/Mitt     57
Region Västra        57
Region Östra         31
                      2

By age:
age
14    123
15    126
16    102
17     63

By sex:
sex
Man       208
Kvinna    202
Annat       4


In [3]:
# Resolve text-only friend wishes via fuzzy name matching
u.resolve_friend_wishes(df_sorted, df)
friend_wishes = u.build_friend_graph(df_sorted)

=== Text Friend Name Matching ===
Text-only wishes: 33
Matched & applied: 24
Generic wishes (not a person): 0
Unresolved (friend not in project): 9

Matched:
  Alida Vakk -> Freja Jernberg (Örnsbergs Scoutkår) [rundresa] via fuzzy(0.93)+kar
  Alma Oliver Elgh -> Elsa Mattsson (Vendelsö Scoutkår) [direktresa] via exact
  Alma Oliver Elgh -> Elsa Mattsson (Vendelsö Scoutkår) [direktresa] via fuzzy(0.92)+kar
  Armilde Westerblom -> Loke Jageteg (Falkenbergs Scoutkår) [rundresa] via exact
  Armilde Westerblom -> Vera Hedgren (Bohus Scoutkår) [rundresa] via exact
  Astrid Dodd -> Gustav Glimtoft (Älvsjö Scoutkår) [direktresa] via exact
  Elsa Mattsson -> Julia Gunnarsson (Vendelsö Scoutkår) [direktresa] via exact
  Freja Caro -> Karin Hugner (S:t Botvids Scoutkår) [direktresa] via exact
  Jona Samuel Söderman -> Olivia Grant (Sundbybergs Scoutkår) [direktresa] via fuzzy(0.75)
  Klara Ivarsson -> Elma Hellberg (Nockeby Scoutkår) [direktresa] via fuzzy(0.81)
  Maja Håkansson -> Sebastian Åstr

In [4]:
# Run the full group assignment algorithm
df_sorted = u.assign_groups(df_sorted, GROUP_SIZE, friend_wishes)
total_groups = df_sorted['group'].nunique()

Participants: 414
Groups: 11 x 36 + 1 x 18 = 12 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 204/235
  Kar violations: 12
  Avg geo spread: 1.5542

=== Phase 2: Fix friend wishes ===
  Swaps: 21
  Friend satisfaction: 235/235
  Kar violations: 11
  Avg geo spread: 1.8101

=== Phase 3: Fix kar violations (geo-aware) ===
  Swaps: 7
  Kar violations: 0
  Friend satisfaction: 225/235
  Avg geo spread: 1.8102

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 4
  Friend satisfaction: 235/235
  Kar violations: 0
  Avg geo spread: 1.8102

=== Phase 4: Diversity optimization (geo-penalized) ===
  Swaps: 309
  Diversity score: 35.10 -> 35.32
  Avg geo spread: 1.8102 -> 2.0946

=== FINAL RESULTS ===
Groups: 11 x 36 + 1 x 18
Friend satisfaction: 235/235 (100%)
Kar violations: 0
Total swaps: 341
Diversity: 35.32
Avg geo spread: 2.0946


In [5]:
# Per-group quality metrics
group_of = df_sorted['group'].values
u.print_group_metrics(df_sorted, group_of, total_groups)

Grupp Storlek  Van% MaxKar     M/K/A  14  15  16  17  AvstandKm Karer
--------------------------------------------------------------------------------
    1      36 12/12      5   15/20/1  12   9  10   5        35    16
    2      36 25/25      4   16/20/0  11  13   8   4       111    17
    3      36 20/20      8   18/17/1  10  14   9   3       113    18
    4      36 22/22      4   15/21/0   9  13   8   6       125    23
    5      36 15/15      5   18/18/0  10  12   7   7        75    23
    6      36 23/23      5   23/13/0   8   9   8  11        98    19
    7      36 22/22      4   15/21/0  12   6  10   8       108    20
    8      36 22/22      6   18/18/0   8  13  10   5       103    20
    9      36 20/20      6   19/15/2  10  10  12   4         9    19
   10      36 19/19      5   22/14/0  13  13   6   4        68    19
   11      36 28/28      8   18/18/0  15   8   9   4        41    12
   12      18   7/7      3    11/7/0   5   6   5   2       148    13
---------------------

In [6]:
# Export CSV + JSON
OUTPUT_DIR = '/config/notebooks/wsj27'
csv_path, json_path = u.export_results(df_sorted, group_of, total_groups, OUTPUT_DIR, prefix='wsj27_direktresa')

Saved 414 participants to /config/notebooks/wsj27/wsj27_direktresa_grupper.csv
Saved group summary to /config/notebooks/wsj27/wsj27_direktresa_grupper.json

CSV preview (first 10 rows):
 group               name member_no  age    sex                kar                   district       region friend_1 friend_2       lat       lng
     1      Edit Fryklund   3351167   15 Kvinna  Bunkeflo Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.557482 12.963234
     1 Rasmus Björn Håkan   3346375   15    Man  Bunkeflo Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.557482 12.963234
     1       Elin Wahlbom   3349984   16 Kvinna     Dalby Scoutkår Södra Skånes Scoutdistrikt Region Södra                   55.664664 13.346159
     1 Lindsey Bäverkvist   3356950   16 Kvinna     Dalby Scoutkår Södra Skånes Scoutdistrikt Region Södra  3451529          55.664664 13.346159
     1          Liv Kropp   3421219   16 Kvinna     Dalby Scoutkår Södra Skånes Scoutdist

In [7]:
# Generate interactive map
map_path = f'{OUTPUT_DIR}/wsj_direktresa_karta.html'
u.generate_group_map_html(df_sorted, total_groups, map_path, title='WSJ 2027 Direktresa',
                          friend_wishes=friend_wishes, show_group_arcs=True)
print(f"\nAll outputs:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Map:  {map_path}")

Friend arcs: 196 (187 satisfied, 9 unsatisfied)
Group arcs: 7083 connections across 12 groups
Saved group map: /config/notebooks/wsj27/wsj_direktresa_karta.html

All outputs:
  CSV:  /config/notebooks/wsj27/wsj27_direktresa_grupper.csv
  JSON: /config/notebooks/wsj27/wsj27_direktresa_grupper.json
  Map:  /config/notebooks/wsj27/wsj_direktresa_karta.html
